# Data integration

This notebook combines transaction, account, customer and merchant data into a single enriched analytical dataset

## Objectives

- Practice the different types of table joins
- Validate primary and foreign keys
- Join transactions with accounts
- Join accounts with customers
- Join transactions with merchants
- Detect unmatched records
- Prevent accidental row duplication
- Create cross table analytical features
- Compare merge, join and concat

In [2]:
from pathlib import Path

import pandas as pd

In [3]:
current_directory = Path.cwd()

In [4]:
if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

project_root

WindowsPath('c:/Users/Focus/Documents/GitHub/transaction-intelligence')

In [5]:
transactions_path = (
    project_root
    / "data"
    / "processed"
    / "transactions_features.csv"
)

accounts_path = (
    project_root
    / "data"
    / "raw"
    / "accounts.csv"
)

customers_path = (
    project_root
    / "data"
    / "raw"
    / "customers.csv"
)

merchants_path = (
    project_root
    / "data"
    / "raw"
    / "merchants.csv"
)

In [7]:
paths_exist = pd.Series(
    {
        "transactions": transactions_path.exists(),
        "accounts": accounts_path.exists(),
        "customers": customers_path.exists(),
        "merchants": merchants_path.exists(),
    },
    name="exists",
)

paths_exist.all()

np.True_

## Loading datasets

The transformed transaction dataset is combined with the three reference tables: accounts, customers and merchants

In [10]:
transactions = pd.read_csv(transactions_path, parse_dates=["timestamp"])
accounts = pd.read_csv(accounts_path, parse_dates=["opening_date"])
customers = pd.read_csv(customers_path, parse_dates=["signup_date"])
merchants = pd.read_csv(merchants_path)

In [11]:
dataset_shapes = pd.DataFrame(
    {
        "rows": [
            len(transactions),
            len(accounts),
            len(customers),
            len(merchants),
        ],
        "columns": [
            len(transactions.columns),
            len(accounts.columns),
            len(customers.columns),
            len(merchants.columns),
        ],
    },
    index=[
        "transactions",
        "accounts",
        "customers",
        "merchants",
    ],
)

dataset_shapes

,rows,columns
transactions,52,24
accounts,16,6
customers,12,5
merchants,15,5


In [13]:
# Before joining tables primary keys are checked for uniqueness and foreign keys are reviewed

key_validation = pd.Series(
    {
        "transaction_id_unique":(transactions["transaction_id"].is_unique),
        "account_id_unique":(accounts["account_id"].is_unique),
        "customer_id_unique":(customers["customer_id"].is_unique),
        "merchant_id_unique":(merchants["merchant_id"].is_unique),
    },
    name = "is_unique"
)

key_validation.all()

np.True_

In [14]:
# Check for columns with repeated names

set(transactions.columns).intersection(accounts.columns)

{'account_id', 'currency', 'status'}

In [15]:
accounts_for_merge = accounts.rename(
    columns = {
        "currency" : "account_currency",
        "status" : "account_status"
    }
)

In [16]:
accounts_for_merge.columns

Index(['account_id', 'customer_id', 'account_type', 'account_currency',
       'opening_date', 'account_status'],
      dtype='str')

In [17]:
customers_for_merge = customers.rename(
    columns = {"country": "customer_country"}
)

In [18]:
merchants_for_merge = merchants.rename(
    columns={
        "category": "merchant_category",
        "country": "merchant_country",
    }
)

In [19]:
merchants_for_merge.columns

Index(['merchant_id', 'merchant_name', 'merchant_category', 'merchant_country',
       'online'],
      dtype='str')

# Transactions and accounts merge

Each transaction belongs to one account while one account may appear in many transactions.

- This is a many to one relationship

In [28]:
transactions_accounts = transactions.merge(
    accounts_for_merge,
    on = "account_id",
    how = "left",
    validate = "many_to_one",
    indicator = "account_merge"
)

In [29]:
transactions.shape

(52, 24)

In [30]:
transactions_accounts.shape

(52, 30)

In [31]:
transactions_accounts["account_merge"
].value_counts(dropna=False)

account_merge
both          52
left_only      0
right_only     0
Name: count, dtype: int64

In [34]:
unmatched_accounts = transactions_accounts.loc[transactions_accounts["account_merge"] != "both", 
["transaction_id", "account_id", "account_merge"]]

unmatched_accounts.empty

True

# Transactions and customers merge

The account table provides the customer_id which is used to add customer information to every transaction

In [35]:
transactions_customers = transactions_accounts.merge(
    customers_for_merge,
    on = "customer_id",
    how = "left",
    validate = "many_to_one",
    indicator = "customer_merge"
)

In [36]:
transactions_customers["customer_merge"].value_counts(dropna = False)

customer_merge
both          52
left_only      0
right_only     0
Name: count, dtype: int64

In [38]:
transactions_customers[
    [
        "transaction_id",
        "customer_id",
        "age",
        "customer_country",
        "signup_date",
        "customer_segment",
    ]
].head()


,transaction_id,customer_id,age,customer_country,signup_date,customer_segment
0,T0001,C001,29,Spain,2023-01-15,Standard
1,T0002,C002,42,Spain,2022-06-03,Premium
2,T0003,C003,23,France,2024-02-11,Student
3,T0004,C004,35,Germany,2021-09-27,Premium
4,T0005,C005,31,Spain,2023-07-19,Standard


# Transactions and merchants merge

Each transaction references one merchant; merchant information is added through merchant_id

In [39]:
transactions_enriched = transactions_customers.merge(
    merchants_for_merge,
    on = "merchant_id",
    how = "left",
    validate = "many_to_one",
    indicator = "merchant_merge"
)

### Transactions audit

In [41]:
merge_audit = pd.Series(
    {
        "original_transaction_rows": len(
            transactions
        ),
        "rows_after_account_merge": len(
            transactions_accounts
        ),
        "rows_after_customer_merge": len(
            transactions_customers
        ),
        "rows_after_merchant_merge": len(
            transactions_enriched
        ),
        "unmatched_accounts": len(
            unmatched_accounts
        ),
        "transaction_ids_remain_unique": (
            transactions_enriched[
                "transaction_id"
            ].is_unique
        ),
    },
    name="value",
)

merge_audit

original_transaction_rows          52
rows_after_account_merge           52
rows_after_customer_merge          52
rows_after_merchant_merge          52
unmatched_accounts                  0
transaction_ids_remain_unique    True
Name: value, dtype: object

In [42]:
transactions_enriched = (transactions_enriched.drop(columns = ["account_merge", "customer_merge", "merchant_merge"]))

In [43]:
transactions_enriched.columns

Index(['transaction_id', 'account_id', 'merchant_id', 'timestamp', 'amount',
       'currency', 'transaction_type', 'status', 'year', 'month', 'month_name',
       'weekday', 'hour', 'absolute_amount', 'transaction_sign',
       'signed_amount', 'net_amount', 'transaction_direction', 'is_weekend',
       'time_period', 'amount_band', 'amount_quartile', 'commission_rate',
       'commission_amount', 'customer_id', 'account_type', 'account_currency',
       'opening_date', 'account_status', 'age', 'customer_country',
       'signup_date', 'customer_segment', 'merchant_name', 'merchant_category',
       'merchant_country', 'online'],
      dtype='str')

# Cross table features

The combined information allows me to create features that could not be calculated from the transaction table alone, such as if the transaction is national or international

In [44]:
transactions_enriched["is_international"] = transactions_enriched["customer_country"].ne(transactions_enriched["merchant_country"])

In [45]:
transactions_enriched[
    [
        "transaction_id",
        "customer_country",
        "merchant_country",
        "is_international",
    ]
].head(15)

,transaction_id,customer_country,merchant_country,is_international
0,T0001,Spain,Spain,False
1,T0002,Spain,Spain,False
2,T0003,France,Spain,True
3,T0004,Germany,Germany,False
4,T0005,Spain,France,True
5,T0006,Italy,Ireland,True
6,T0008,Spain,Netherlands,True
7,T0009,Netherlands,Portugal,True
8,T0010,Ireland,Spain,True
9,T0011,Spain,Italy,True


In [46]:
transactions_enriched["transaction_scope"] = (transactions_enriched["is_international"].map(
    {
        False: "Domestic",
        True: "International"
    }
))

In [47]:
transactions_enriched[
    "transaction_scope"
].value_counts()

transaction_scope
International    37
Domestic         15
Name: count, dtype: int64

In [ ]:
transactions_enriched["currency_matches_account"] = (
    transactions_enriched["currency"]
    .eq(
        transactions_enriched[
            "account_currency"
        ]
    )
)

In [50]:
transactions_enriched[
    [
        "transaction_id",
        "customer_id",
        "customer_segment",
        "customer_country",
        "account_id",
        "account_type",
        "merchant_name",
        "merchant_category",
        "merchant_country",
        "transaction_scope",
        "amount",
        "net_amount",
        "commission_amount",
    ]
].head(10)

,transaction_id,customer_id,customer_segment,customer_country,account_id,account_type,merchant_name,merchant_category,merchant_country,transaction_scope,amount,net_amount,commission_amount
0,T0001,C001,Standard,Spain,A001,Current,Green Market,Groceries,Spain,Domestic,34.80,34.80,0.52
1,T0002,C002,Premium,Spain,A003,Current,Urban Coffee,Restaurants,Spain,Domestic,4.20,4.20,0.06
2,T0003,C003,Student,France,A005,Current,Book Planet,Books,Spain,International,18.99,18.99,0.28
3,T0004,C004,Premium,Germany,A006,Current,Tech World,Electronics,Germany,Domestic,249.90,249.90,3.75
4,T0005,C005,Standard,Spain,A008,Current,Fresh Basket,Groceries,France,International,62.40,62.40,0.94
5,T0006,C006,Premium,Italy,A009,Current,Cloud Store,Electronics,Ireland,International,89.00,89.00,1.34
6,T0008,C008,Standard,Spain,A012,Current,Travel Now,Travel,Netherlands,International,320.00,320.00,4.80
7,T0009,C009,Student,Netherlands,A013,Current,Home Corner,Home,Portugal,International,74.50,74.50,1.12
8,T0010,C010,Premium,Ireland,A014,Current,Cinema Central,Entertainment,Spain,International,13.00,13.00,0.20
9,T0011,C011,Standard,Spain,A015,Current,Style Avenue,Fashion,Italy,International,96.75,96.75,1.45


## join() demonstration

join() is good when the lookup key is stored in the index of the right hand table. I will not use it to build the final result but I will demonstrate its use

In [51]:
merchant_name_lookup = (merchants_for_merge.set_index("merchant_id")["merchant_name"])

In [52]:
join_example = (transactions[["transaction_id", "merchant_id"]]
    .set_index("merchant_id")
    .join(merchant_name_lookup, how = "left")
    .reset_index()
)
join_example.head()

,merchant_id,transaction_id,merchant_name
0,M001,T0001,Green Market
1,M002,T0002,Urban Coffee
2,M003,T0003,Book Planet
3,M004,T0004,Tech World
4,M005,T0005,Fresh Basket


# Integration validation

The enriched dataset is validated before being saved

In [54]:
integration_validation = pd.Series(
    {
        "row_count_preserved": (
            len(transactions_enriched)
            == len(transactions)
        ),
        "transaction_ids_are_unique": (
            transactions_enriched[
                "transaction_id"
            ].is_unique
        ),
        "customer_ids_are_present": (
            transactions_enriched[
                "customer_id"
            ].notna().all()
        ),
        "merchant_names_are_present": (
            transactions_enriched[
                "merchant_name"
            ].notna().all()
        ),
        "customer_countries_are_present": (
            transactions_enriched[
                "customer_country"
            ].notna().all()
        ),
        "merchant_countries_are_present": (
            transactions_enriched[
                "merchant_country"
            ].notna().all()
        ),
        "transaction_scope_is_present": (
            transactions_enriched[
                "transaction_scope"
            ].notna().all()
        ),
        "no_complete_rows_duplicated": (
            ~transactions_enriched
            .duplicated()
            .any()
        ),
    },
    name="passed",
)

integration_validation.all()

np.True_

In [56]:
transactions_enriched.shape

(52, 40)

In [57]:
transactions_enriched.columns.tolist()

['transaction_id',
 'account_id',
 'merchant_id',
 'timestamp',
 'amount',
 'currency',
 'transaction_type',
 'status',
 'year',
 'month',
 'month_name',
 'weekday',
 'hour',
 'absolute_amount',
 'transaction_sign',
 'signed_amount',
 'net_amount',
 'transaction_direction',
 'is_weekend',
 'time_period',
 'amount_band',
 'amount_quartile',
 'commission_rate',
 'commission_amount',
 'customer_id',
 'account_type',
 'account_currency',
 'opening_date',
 'account_status',
 'age',
 'customer_country',
 'signup_date',
 'customer_segment',
 'merchant_name',
 'merchant_category',
 'merchant_country',
 'online',
 'is_international',
 'transaction_scope',
 'currency_matches_account']

In [59]:
transactions_enriched = (
    transactions_enriched.sort_values(
        ["customer_id", "timestamp"]
    )
    .reset_index(drop = True)
)

In [60]:
# Save enriched table
enriched_transactions_path = (
    project_root
    / "data"
    / "processed"
    / "transactions_enriched.csv"
)

In [61]:
transactions_enriched.to_csv(
    enriched_transactions_path,
    index=False,
)

In [63]:
enriched_transactions_path.exists()

True

## Integration results

The transformed transaction data was successfully combined with account, customer and merchant information

The integration process:

- Validated all primary keys before joining
- Used many_to_one validation to prevent accidental row multiplication
- Preserved all transaction rows with left joins
- Used merge indicators to identify unmatched records
- Confirmed that every account, customer and merchant reference was matched
- Renamed overlapping columns to preserve clear meanings
- Created domestic and international transaction indicators
- Compared transaction currency with account currency

The enriched dataset contains one row per transaction and includes the contextual information required for customer, merchant and anomaly analysis